# World Cup Project — Get Data Starter Notebook

This notebook downloads historical international soccer data and saves it into a local `data/` folder.

It creates both raw and cleaned CSV files that can be used in the modeling notebook.


## What data are we getting?

This notebook pulls three public CSV files from the `international_results` GitHub dataset:

1. **Match results**
   - Historical international match results
   - Includes teams, scores, dates, tournament names, match location, and whether the match was played on neutral ground
   - This is the main dataset for modeling match outcomes

2. **Shootouts**
   - Penalty shootout results
   - Useful later if you build a knockout-round World Cup simulator

3. **Goalscorers**
   - Player-level goalscorer data
   - Includes scorer names, minute of goal, own goals, and penalties
   - Not required for the first match model, but useful for later player/team attacking analysis


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

pd.set_option("display.max_columns", 100)

print("Current working directory:")
print(os.getcwd())


## 1. Create the data folder

This creates a `data/` folder in the same working directory where your notebook is running.


In [ ]:
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Data folder:")
print(DATA_DIR.resolve())


## 2. Download the raw datasets

In [ ]:
results_url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
shootouts_url = "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv"
goalscorers_url = "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv"

results = pd.read_csv(results_url)
shootouts = pd.read_csv(shootouts_url)
goalscorers = pd.read_csv(goalscorers_url)

print("Downloaded rows:")
print("results:", len(results))
print("shootouts:", len(shootouts))
print("goalscorers:", len(goalscorers))


## 3. Preview match results

This is the most important dataset for the first modeling notebook.


In [ ]:
results.head()


In [ ]:
results.info()


## 4. Preview shootouts

In [ ]:
shootouts.head()


## 5. Preview goalscorers

In [ ]:
goalscorers.head()


## 6. Save raw CSV files

These are saved exactly as downloaded.


In [ ]:
results.to_csv(DATA_DIR / "results_raw.csv", index=False)
shootouts.to_csv(DATA_DIR / "shootouts_raw.csv", index=False)
goalscorers.to_csv(DATA_DIR / "goalscorers_raw.csv", index=False)

print("Raw files saved:")
print(DATA_DIR / "results_raw.csv")
print(DATA_DIR / "shootouts_raw.csv")
print(DATA_DIR / "goalscorers_raw.csv")


## 7. Clean the match results

For the starter model, we need clean match-level data.

Cleaning steps:
- Convert `date` to datetime
- Remove rows missing scores or team names
- Standardize team and tournament text columns
- Ensure scores are numeric integers
- Ensure `neutral` is a true/false column
- Create result labels:
  - `home_win`
  - `draw`
  - `away_win`
- Create basic scoring columns:
  - `goal_diff`
  - `total_goals`


In [ ]:
results_clean = results.copy()

results_clean["date"] = pd.to_datetime(results_clean["date"], errors="coerce")

text_cols = ["home_team", "away_team", "tournament", "city", "country"]
for col in text_cols:
    if col in results_clean.columns:
        results_clean[col] = results_clean[col].astype(str).str.strip()

results_clean["home_score"] = pd.to_numeric(results_clean["home_score"], errors="coerce")
results_clean["away_score"] = pd.to_numeric(results_clean["away_score"], errors="coerce")

results_clean = results_clean.dropna(
    subset=["date", "home_team", "away_team", "home_score", "away_score"]
).copy()

results_clean["home_score"] = results_clean["home_score"].astype(int)
results_clean["away_score"] = results_clean["away_score"].astype(int)

if "neutral" in results_clean.columns:
    results_clean["neutral"] = results_clean["neutral"].astype(bool)
else:
    results_clean["neutral"] = False

results_clean["result"] = np.select(
    [
        results_clean["home_score"] > results_clean["away_score"],
        results_clean["home_score"] == results_clean["away_score"],
        results_clean["home_score"] < results_clean["away_score"],
    ],
    ["home_win", "draw", "away_win"]
)

results_clean["goal_diff"] = results_clean["home_score"] - results_clean["away_score"]
results_clean["total_goals"] = results_clean["home_score"] + results_clean["away_score"]

results_clean = results_clean.sort_values("date").reset_index(drop=True)

results_clean.head()


## 8. Clean shootouts

Basic cleaning:
- Convert `date` to datetime
- Strip team names
- Sort by date


In [ ]:
shootouts_clean = shootouts.copy()

shootouts_clean["date"] = pd.to_datetime(shootouts_clean["date"], errors="coerce")

for col in ["home_team", "away_team", "winner"]:
    if col in shootouts_clean.columns:
        shootouts_clean[col] = shootouts_clean[col].astype(str).str.strip()

shootouts_clean = shootouts_clean.dropna(subset=["date"]).copy()
shootouts_clean = shootouts_clean.sort_values("date").reset_index(drop=True)

shootouts_clean.head()


## 9. Clean goalscorers

Basic cleaning:
- Convert `date` to datetime
- Strip text columns
- Convert minute to numeric
- Ensure own-goal and penalty fields are true/false


In [ ]:
goalscorers_clean = goalscorers.copy()

goalscorers_clean["date"] = pd.to_datetime(goalscorers_clean["date"], errors="coerce")

for col in ["home_team", "away_team", "team", "scorer"]:
    if col in goalscorers_clean.columns:
        goalscorers_clean[col] = goalscorers_clean[col].astype(str).str.strip()

if "minute" in goalscorers_clean.columns:
    goalscorers_clean["minute"] = pd.to_numeric(goalscorers_clean["minute"], errors="coerce")

for col in ["own_goal", "penalty"]:
    if col in goalscorers_clean.columns:
        goalscorers_clean[col] = goalscorers_clean[col].astype(bool)

goalscorers_clean = goalscorers_clean.dropna(subset=["date"]).copy()
goalscorers_clean = goalscorers_clean.sort_values("date").reset_index(drop=True)

goalscorers_clean.head()


## 10. Save cleaned CSV files

This is the key section.

The modeling notebook will look for `data/results.csv`, so this notebook saves the cleaned match results under that exact name.

It also saves `results_clean.csv` for clarity.


In [ ]:
results_clean.to_csv(DATA_DIR / "results.csv", index=False)
results_clean.to_csv(DATA_DIR / "results_clean.csv", index=False)

shootouts_clean.to_csv(DATA_DIR / "shootouts.csv", index=False)
shootouts_clean.to_csv(DATA_DIR / "shootouts_clean.csv", index=False)

goalscorers_clean.to_csv(DATA_DIR / "goalscorers.csv", index=False)
goalscorers_clean.to_csv(DATA_DIR / "goalscorers_clean.csv", index=False)

print("Cleaned files saved to:")
print(DATA_DIR.resolve())

print("\nSaved CSV files:")
for file in sorted(DATA_DIR.glob("*.csv")):
    print("-", file.name)


## 11. Verify the files exist

Run this cell to make sure the CSVs were actually saved where you expect.


In [ ]:
expected_files = [
    "results.csv",
    "results_clean.csv",
    "shootouts.csv",
    "shootouts_clean.csv",
    "goalscorers.csv",
    "goalscorers_clean.csv",
]

for filename in expected_files:
    path = DATA_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")


## 12. Quick data summary

In [ ]:
print("Match results date range:")
print(results_clean["date"].min(), "to", results_clean["date"].max())

print("\nNumber of matches:", len(results_clean))
print("Number of teams:", pd.concat([results_clean["home_team"], results_clean["away_team"]]).nunique())

print("\nResult distribution:")
print(results_clean["result"].value_counts(normalize=True).round(3))


## Next step

After this notebook saves the CSVs, run the modeling notebook.

The modeling notebook should automatically find:

```text
data/results.csv
```
